In [36]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

rho, S, c = 1.225, 0.5, 0.3
V     = np.array([50.0, 60.0, 70.0])
alpha = np.deg2rad([2.0, 6.0, 10.0])
de    = np.deg2rad([-1.0, 2.0, 5.0])
M     = np.array([8.2773, -5.9759, -32.4914])

q  = 0.5*rho*V**2          # Task 1
Cm = M/(q*S*c)             # Task 2

for i in range(3):
    print(f"Test {i+1}: q = {q[i]:10.4f} Pa,  Cm = {Cm[i]: .6f}")

Test 1: q =  1531.2500 Pa,  Cm =  0.036037
Test 2: q =  2205.0000 Pa,  Cm = -0.018068
Test 3: q =  3001.2500 Pa,  Cm = -0.072173


In [37]:
A = np.column_stack([np.ones(3), alpha, de])
b = Cm.copy()

for i in range(3):
    print(f"Eq {i+1}: Cm0 + ({alpha[i]:.6f}) Cmα + ({de[i]:.6f}) Cmδe = {b[i]:.6f}")
print("\nA =\n", A)
print("b =", b)

Eq 1: Cm0 + (0.034907) Cmα + (-0.017453) Cmδe = 0.036037
Eq 2: Cm0 + (0.104720) Cmα + (0.034907) Cmδe = -0.018068
Eq 3: Cm0 + (0.174533) Cmα + (0.087266) Cmδe = -0.072173

A =
 [[ 1.        0.034907 -0.017453]
 [ 1.        0.10472   0.034907]
 [ 1.        0.174533  0.087266]]
b = [ 0.036037 -0.018068 -0.072173]


Cm0=C0[0]
Cma=C0[1]
Cmd=C0[2]

Cm[0] = Cm0 + Cma*2 + Cmd*(-1)
Cm[1] = Cm0 + Cma*6 + Cmd*(2)
Cm[2] = Cm0 + Cma*10 + Cmd*(5)



In [38]:
TOL = 1e-10

def crout(A, verbose=True):
    n = A.shape[0]
    L = np.zeros((n, n)); U = np.eye(n)
    for j in range(n):
        for i in range(j, n):                      # column j of L
            s = sum(L[i, k]*U[k, j] for k in range(j))
            L[i, j] = A[i, j] - s
            if verbose:
                print(f"L[{i+1},{j+1}] = A[{i+1},{j+1}] - {s:.6f} = {L[i,j]:.6e}")
        if abs(L[j, j]) < TOL:
            print(f">>> pivot L[{j+1},{j+1}] = {L[j,j]:.2e} ~ 0  ->  A is SINGULAR")
            continue                               
        for i in range(j+1, n):                    
            s = sum(L[j, k]*U[k, i] for k in range(j))
            U[j, i] = (A[j, i] - s)/L[j, j]
            if verbose:
                print(f"U[{j+1},{i+1}] = (A[{j+1},{i+1}] - {s:.6f}) / L[{j+1},{j+1}] = {U[j,i]:.6f}")
        if verbose: print("-"*50)
    return L, U

In [39]:
L, U = crout(A)
print("\nL =\n", L)
print("\nU =\n", U)
print("det(A) =", np.linalg.det(A), "  rank(A) =", np.linalg.matrix_rank(A, tol=1e-10))

L[1,1] = A[1,1] - 0.000000 = 1.000000e+00
L[2,1] = A[2,1] - 0.000000 = 1.000000e+00
L[3,1] = A[3,1] - 0.000000 = 1.000000e+00
U[1,2] = (A[1,2] - 0.000000) / L[1,1] = 0.034907
U[1,3] = (A[1,3] - 0.000000) / L[1,1] = -0.017453
--------------------------------------------------
L[2,2] = A[2,2] - 0.034907 = 6.981317e-02
L[3,2] = A[3,2] - 0.034907 = 1.396263e-01
U[2,3] = (A[2,3] - -0.017453) / L[2,2] = 0.750000
--------------------------------------------------
L[3,3] = A[3,3] - 0.087266 = 1.387779e-17
>>> pivot L[3,3] = 1.39e-17 ~ 0  ->  A is SINGULAR

L =
 [[1.       0.       0.      ]
 [1.       0.069813 0.      ]
 [1.       0.139626 0.      ]]

U =
 [[ 1.        0.034907 -0.017453]
 [ 0.        1.        0.75    ]
 [ 0.        0.        1.      ]]
det(A) = 9.68852360557951e-19   rank(A) = 2


In [40]:
def forward_sub(L, b, verbose=True):
    n = len(b); y = np.zeros(n)
    for i in range(n):
        s = sum(L[i, k]*y[k] for k in range(i))
        if abs(L[i, i]) < TOL:
            resid = b[i] - s
            print(f"y[{i+1}]: 0*y = {resid:.2e}  -> consistent (~0), y[{i+1}] is FREE" if abs(resid) < 1e-5
                  else f"y[{i+1}]: 0*y = {resid:.2e}  -> INCONSISTENT, no solution")
            y[i] = np.nan                           # free parameter
        else:
            y[i] = (b[i] - s)/L[i, i]
            if verbose: print(f"y[{i+1}] = ({b[i]:.6f} - {s:.6f}) / {L[i,i]:.6f} = {y[i]:.6f}")
    return y

y = forward_sub(L, b)

y[1] = (0.036037 - 0.000000) / 1.000000 = 0.036037
y[2] = (-0.018068 - 0.036037) / 0.069813 = -0.774996
y[3]: 0*y = -3.65e-07  -> consistent (~0), y[3] is FREE


In [41]:
def backward_sub(U, y, t):
    n = len(y); x = np.zeros(n)
    y = np.where(np.isnan(y), t, y)                 # free parameter
    for i in range(n-1, -1, -1):
        s = sum(U[i, k]*x[k] for k in range(i+1, n))
        x[i] = (y[i] - s)/U[i, i]
    return x

# Family of solutions: x(t) = x(0) + t * d   (d = null-space direction)
x0 = backward_sub(U, y, 0.0)
d  = backward_sub(U, y, 1.0) - x0
print("Cm0   = %.6f + %.6f * t" % (x0[0], d[0]))
print("Cmα   = %.6f + %.6f * t" % (x0[1], d[1]))
print("Cmδe  = t   (free)")
print("null-space check  A@d =", A @ d)

# Pick t = Cmδe = -0.3 (gives the 'clean' values Cm0=0.05, Cmα=-0.55)
t = -0.3
x = backward_sub(U, y, t)
Cm0, Cma, Cmde = x
print(f"\nWith t = {t}:")
print(f"Cm0  = {Cm0:.6f}")
print(f"Cmα  = {Cma:.6f} per rad")
print(f"Cmδe = {Cmde:.6f} per rad")

Cm0   = 0.063090 + 0.043633 * t
Cmα   = -0.774996 + -0.750000 * t
Cmδe  = t   (free)
null-space check  A@d = [0. 0. 0.]

With t = -0.3:
Cm0  = 0.050000
Cmα  = -0.549996 per rad
Cmδe = -0.300000 per rad


In [42]:
R = A - L @ U
print("A - LU =\n", R)
print("||A - LU||_inf =", np.linalg.norm(R, np.inf))
print("Residual of Ax-b :", A @ x - b)

A - LU =
 [[ 0.  0.  0.]
 [ 0.  0. -0.]
 [ 0.  0.  0.]]
||A - LU||_inf = 6.938893903907228e-18
Residual of Ax-b : [0. 0. 0.]


In [43]:
a_p, d_p = np.deg2rad(8.0), np.deg2rad(3.0)
Cm_pred = Cm0 + Cma*a_p + Cmde*d_p
print(f"Predicted Cm (t = {t}) = {Cm_pred:.6f}")

print("\nSensitivity to the free parameter:")
for tt in [-0.6, -0.3, 0.0, 0.3]:
    xx = backward_sub(U, y, tt)
    print(f"  Cmδe = {tt:5.1f}  ->  Cm_pred = {xx[0] + xx[1]*a_p + xx[2]*d_p: .6f}")

Predicted Cm (t = -0.3) = -0.042502

Sensitivity to the free parameter:
  Cmδe =  -0.6  ->  Cm_pred = -0.039884
  Cmδe =  -0.3  ->  Cm_pred = -0.042502
  Cmδe =   0.0  ->  Cm_pred = -0.045120
  Cmδe =   0.3  ->  Cm_pred = -0.047738


Question 2

In [44]:
CL = np.array([0.15, 0.35, 0.55, 0.85])
CD = np.array([0.01934625, 0.02464125, 0.03437625, 0.05835375])

# CD = CD0 + k1*CL + k2*CL^2 + k3*CL^3
A = np.array([[1, CL[0], CL[0]**2, CL[0]**3],
              [1, CL[1], CL[1]**2, CL[1]**3],
              [1, CL[2], CL[2]**2, CL[2]**3],
              [1, CL[3], CL[3]**2, CL[3]**3]])
b = CD
print(A)

[[1.       0.15     0.0225   0.003375]
 [1.       0.35     0.1225   0.042875]
 [1.       0.55     0.3025   0.166375]
 [1.       0.85     0.7225   0.614125]]


In [45]:
L, U = crout(A)
print("L =\n", L)
print("U =\n", U)

L[1,1] = A[1,1] - 0.000000 = 1.000000e+00
L[2,1] = A[2,1] - 0.000000 = 1.000000e+00
L[3,1] = A[3,1] - 0.000000 = 1.000000e+00
L[4,1] = A[4,1] - 0.000000 = 1.000000e+00
U[1,2] = (A[1,2] - 0.000000) / L[1,1] = 0.150000
U[1,3] = (A[1,3] - 0.000000) / L[1,1] = 0.022500
U[1,4] = (A[1,4] - 0.000000) / L[1,1] = 0.003375
--------------------------------------------------
L[2,2] = A[2,2] - 0.150000 = 2.000000e-01
L[3,2] = A[3,2] - 0.150000 = 4.000000e-01
L[4,2] = A[4,2] - 0.150000 = 7.000000e-01
U[2,3] = (A[2,3] - 0.022500) / L[2,2] = 0.500000
U[2,4] = (A[2,4] - 0.003375) / L[2,2] = 0.197500
--------------------------------------------------
L[3,3] = A[3,3] - 0.222500 = 8.000000e-02
L[4,3] = A[4,3] - 0.372500 = 3.500000e-01
U[3,4] = (A[3,4] - 0.082375) / L[3,3] = 1.050000
--------------------------------------------------
L[4,4] = A[4,4] - 0.509125 = 1.050000e-01
--------------------------------------------------
L =
 [[1.    0.    0.    0.   ]
 [1.    0.2   0.    0.   ]
 [1.    0.4   0.08  0. 

In [46]:
n = 4

# Ly = b
y = np.zeros(n)
for i in range(n):
    y[i] = (b[i] - np.sum(L[i, :i] * y[:i])) / L[i, i]
print("y =", y)

# Ux = y
x = np.zeros(n)
for i in range(n-1, -1, -1):
    x[i] = (y[i] - np.sum(U[i, i+1:] * x[i+1:])) / U[i, i]
print("CD0, k1, k2, k3 =", x)

y = [0.019346 0.026475 0.0555   0.01    ]
CD0, k1, k2, k3 = [0.018 0.002 0.045 0.01 ]


In [47]:
R = A - L @ U
print("R =\n", R)
print("||R||inf =", np.linalg.norm(R, np.inf))

er = np.linalg.norm(A @ x - b, np.inf) / np.linalg.norm(b, np.inf)
print("relative residual =", er)

R =
 [[ 0.  0.  0.  0.]
 [ 0.  0.  0. -0.]
 [ 0.  0.  0.  0.]
 [ 0.  0.  0.  0.]]
||R||inf = 2.0816681711721685e-17
relative residual = 1.1891084812727935e-16
